In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict    
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0

)

In [3]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [4]:
def generate_joke(state: JokeState):
    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):
    prompt = f'Write an explantion of the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation' : response}

In [7]:
# Graph Building

graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config2 = {"configurable": {"thread_id" : "2"}}
workflow.invoke({'topic' : 'Pasta'},config=config2)

d:\VS code\Python\Langgraph\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\VS code\Python\Langgraph\venv\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'topic': 'Pasta',
 'joke': [{'type': 'text',
   'text': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little **cannoli**.',
   'extras': {'signature': 'EnsKeQFpFH0ToDYaEGZkGP+xbLv4Oq3Hnb3IQvCgh06OVZ4DzbOxLWDvN+lGFMfavmSsM6WPewU/RhNK6Wag/ATcmnX+1SYpwNJAqLLLqfv5z5eeDvn9zvU7UuP2bcUL8T6oslWvI9yO/UNMAYqpijjPjuzkGWSIQlQNhgA='}}],
 'explanation': [{'type': 'text',
   'text': 'This joke relies on a **pun**, which is a joke exploiting different possible meanings of a word or the fact that words sound alike.\n\nHere is the breakdown of the humor:\n\n1. **The Setup ("Why did the pasta go to the doctor?"):** This creates an expectation that the pasta is sick or injured, which is a standard setup for a classic joke. \n\n2. **The Punchline ("Because it was feeling a little *cannoli*"):** \n   * **The Sound-Alike:** The word *"cannoli"* (a type of Italian pastry filled with sweet ricotta) is used here to sound like the phrase **"can only"** or **"a little lonely."** \n   * **The D

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Pasta', 'joke': [{'type': 'text', 'text': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little **cannoli**.', 'extras': {'signature': 'EnsKeQFpFH0ToDYaEGZkGP+xbLv4Oq3Hnb3IQvCgh06OVZ4DzbOxLWDvN+lGFMfavmSsM6WPewU/RhNK6Wag/ATcmnX+1SYpwNJAqLLLqfv5z5eeDvn9zvU7UuP2bcUL8T6oslWvI9yO/UNMAYqpijjPjuzkGWSIQlQNhgA='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun**, which is a joke exploiting different possible meanings of a word or the fact that words sound alike.\n\nHere is the breakdown of the humor:\n\n1. **The Setup ("Why did the pasta go to the doctor?"):** This creates an expectation that the pasta is sick or injured, which is a standard setup for a classic joke. \n\n2. **The Punchline ("Because it was feeling a little *cannoli*"):** \n   * **The Sound-Alike:** The word *"cannoli"* (a type of Italian pastry filled with sweet ricotta) is used here to sound like the phrase **"can only"** or **"a little lonely."** \n  

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Pasta', 'joke': [{'type': 'text', 'text': 'Why did the pasta go to the doctor?\n\nBecause it was feeling a little **cannoli**.', 'extras': {'signature': 'EnsKeQFpFH0ToDYaEGZkGP+xbLv4Oq3Hnb3IQvCgh06OVZ4DzbOxLWDvN+lGFMfavmSsM6WPewU/RhNK6Wag/ATcmnX+1SYpwNJAqLLLqfv5z5eeDvn9zvU7UuP2bcUL8T6oslWvI9yO/UNMAYqpijjPjuzkGWSIQlQNhgA='}}], 'explanation': [{'type': 'text', 'text': 'This joke relies on a **pun**, which is a joke exploiting different possible meanings of a word or the fact that words sound alike.\n\nHere is the breakdown of the humor:\n\n1. **The Setup ("Why did the pasta go to the doctor?"):** This creates an expectation that the pasta is sick or injured, which is a standard setup for a classic joke. \n\n2. **The Punchline ("Because it was feeling a little *cannoli*"):** \n   * **The Sound-Alike:** The word *"cannoli"* (a type of Italian pastry filled with sweet ricotta) is used here to sound like the phrase **"can only"** or **"a little lonely."** \n 